# Cyber Defense Lab

## Defensive Security Studio

**Author:** P.Vaishnavi  
**Website:** `https://cyber-defense-lab-site.vercel.app/`  
**Purpose:** A Cybersecurity Bootcamp-ready defensive cybersecurity capstone with a standalone dashboard and supporting security notebook.

The website is a self-contained HTML/CSS/JavaScript dashboard. It runs locally or as a static GitHub Pages site; no backend or build step is required. The notebook documents the same 12 modules and can export eight evidence tables.

All active checks are opt-in and intended only for systems that you own or are explicitly authorised to assess. The notebook does not perform exploitation, credential testing, persistence, or destructive actions. Its web audit and optional machine-learning download are disabled by default.

## Project objectives

The dashboard and notebook cover the same 12 defensive modules:

1. Active Directory
2. Web Scanner
3. API Security
4. Enterprise SIEM
5. SOAR Playbook
6. ML Network Intrusion Detector
7. Cloud IAM Entitlement Audit
8. DevSecOps CI/CD Pipeline
9. Zero-Trust Network Access
10. Enterprise Ransomware Assessment
11. STRIDE Threat Model
12. Threat Intelligence Aggregator

The notebook provides checklists, plans, demo evidence, and opt-in authorised checks. A module plan is not a claim that a full production assessment has run.

## Companion website and deployment

- `CyberDefenseLab_Vaishnavi.html` is the standalone source file. It uses inline CSS and JavaScript only.
- For a GitHub Pages root homepage, copy the standalone file to `index.html` after extraction; the submission ZIP contains only the five requested files.
- The website's security metrics are illustrative dashboard values. The notebook exports eight CSV evidence tables when its export cell runs.
- The browser scanner sends one GET request without credentials or cookies. Browser CORS rules can block the response or hide headers. The notebook scanner runs separately with Python `requests` and is disabled by default.
- GitHub Pages hosts only the static website. Run this notebook locally or in an authorised Jupyter environment; Pages does not execute notebooks.
- Git setup, push, and Pages instructions are in `README.md`.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
from urllib.parse import urlparse

import pandas as pd
import requests
from urllib.parse import quote

PROJECT_ROOT = Path.cwd()
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)

print(f"Reports will be saved to: {REPORT_DIR.resolve()}")

In [ ]:
module_scope_matrix = pd.DataFrame([
    ["Active Directory", "Checklist", "Authorised directory exports or lab domain"],
    ["Web Scanner", "Single GET header review", "Authorised HTTP(S) URL; response headers"],
    ["API Security", "BOLA test plan", "Authorised API endpoint and test identities"],
    ["Enterprise SIEM", "Detection rules", "Exported logs and detection rules"],
    ["SOAR Playbook", "Approval-gated response plan", "Authorised playbooks and runbooks"],
    ["ML Network Intrusion Detector", "Optional model baseline", "Authorised flow data or benchmark"],
    ["Cloud IAM Entitlement Audit", "Policy review template", "Authorised cloud configuration evidence"],
    ["DevSecOps CI/CD Pipeline", "Pipeline control template", "Authorised pipeline and build evidence"],
    ["Zero-Trust Network Access", "Access-control checklist", "Authorised identity, device, and policy evidence"],
    ["Enterprise Ransomware Assessment", "Resilience risk register", "Authorised backup and recovery evidence"],
    ["STRIDE Threat Model", "Threat and mitigation matrix", "Architecture, data flows, assets, and trust boundaries"],
    ["Threat Intelligence Aggregator", "Indicator normalisation", "Authorised IoC sources; credentials stored outside the notebook"],
], columns=["module", "approach", "evidence_or_authorisation_required"])

assert module_scope_matrix["module"].nunique() == 12
module_scope_matrix


## 1. Active Directory hardening checklist

The original lab idea mentions Kerberoasting. In this submission, the focus is on reducing the conditions that make credential abuse more likely and on monitoring the relevant Windows events. The checklist is suitable for a disposable lab domain or a controlled review.

In [ ]:
ad_hardening_checklist = pd.DataFrame([
    ["Service accounts", "Use gMSA where possible; block interactive logon", "High", "Identity team"],
    ["Kerberos", "Review SPNs and monitor unusual service-ticket volume", "High", "Identity team"],
    ["Privileged groups", "Use separate admin accounts and time-bound membership", "High", "Identity team"],
    ["LDAP", "Require signing and channel binding where compatible", "High", "Domain team"],
    ["SMB", "Require signing and remove legacy SMBv1", "High", "Endpoint team"],
    ["Group Policy", "Enable auditing, PowerShell logging, Defender, firewall, and LAPS", "High", "Endpoint team"],
    ["Delegation", "Review unconstrained and unnecessary delegation", "High", "Identity team"],
    ["Monitoring", "Forward logon, Kerberos, and group-change events to the SIEM", "Medium", "SOC"],
], columns=["control_area", "expected_control", "priority", "owner"])

ad_hardening_checklist

In [ ]:
ad_validation_questions = [
    "Are service accounts prevented from interactive logon?",
    "Are privileged group changes approved and alerted?",
    "Is LDAP signing enforced on domain controllers and clients?",
    "Are LAPS-managed local administrator passwords deployed?",
    "Are unusual Kerberos service-ticket patterns reviewed?",
    "Are domain-controller audit logs retained centrally?",
]

for question in ad_validation_questions:
    print(f"- {question}")

## 2. Web Scanner — HTTP security headers

The website makes one browser-side GET request and checks six defensive headers plus wildcard CORS. It does not crawl or submit data. Browser CORS may block response access or hide headers. The notebook uses Python `requests` in an authorised environment; its live audit is disabled by default.

In [ ]:
SECURITY_HEADERS = {
    "strict-transport-security": "Enforce HTTPS and protect against downgrade attacks.",
    "content-security-policy": "Reduce the impact of script and style injection.",
    "x-content-type-options": "Prevent MIME sniffing.",
    "referrer-policy": "Limit referrer data leakage.",
    "permissions-policy": "Restrict powerful browser features.",
    "x-frame-options": "Reduce clickjacking risk, or use CSP frame-ancestors.",
}


def audit_web_target(url: str, timeout: int = 8) -> dict:
    """Return a small defensive report for an authorised HTTP(S) target."""
    parsed = urlparse(url)
    if (parsed.scheme not in {"http", "https"} or not parsed.hostname
            or parsed.username or parsed.password):
        raise ValueError("Use an http(s) URL without embedded credentials")

    with requests.get(
        url,
        timeout=timeout,
        allow_redirects=False,
        headers={"User-Agent": "CyberDefenseLab/1.0"},
        auth=None,
        cookies=None,
        stream=True,
    ) as response:
        headers = {key.lower(): value for key, value in response.headers.items()}
        findings = []

        for header, purpose in SECURITY_HEADERS.items():
            if header not in headers:
                findings.append({
                    "severity": "high" if header == "strict-transport-security" else "medium",
                    "finding": f"Missing {header}",
                    "remediation": purpose,
                })

        allow_origin = headers.get("access-control-allow-origin", "").strip()
        allow_credentials = headers.get("access-control-allow-credentials", "").lower()
        if allow_origin == "*":
            findings.append({
                "severity": "medium",
                "finding": "Wildcard Access-Control-Allow-Origin",
                "remediation": "Restrict cross-origin access to trusted origins where appropriate.",
            })
        if allow_origin == "*" and allow_credentials == "true":
            findings.append({
                "severity": "high",
                "finding": "Wildcard CORS is combined with credentials",
                "remediation": "Validate Origin server-side and avoid wildcard credentials.",
            })

        return {
            "url": url,
            "final_url": response.url,
            "status_code": response.status_code,
            "tls_used": response.url.startswith("https://"),
            "findings": findings,
            "observed_headers": headers,
        }


RUN_WEB_AUDIT = False
AUTHORIZED_TARGET = "https://your-authorised-lab.example/"

if RUN_WEB_AUDIT:
    web_result = audit_web_target(AUTHORIZED_TARGET)
    print(pd.DataFrame(web_result["findings"]).to_string(index=False))
else:
    print("Web audit is disabled. Set RUN_WEB_AUDIT=True only for an authorised target.")

## 3. API security: BOLA test design

BOLA is tested by comparing two authorised test identities against the same object. The matrix below describes expected behaviour without sending requests. It can be copied into a Postman collection or used to define an authenticated ZAP context in a staging environment.

In [ ]:
def build_bola_matrix(base_url: str, object_ids=("1001", "1002")) -> pd.DataFrame:
    rows = []
    for object_id in object_ids:
        rows.extend([
            {"case": f"owner_reads_{object_id}", "method": "GET", "path": f"/api/orders/{object_id}", "actor": "owner", "expected": "200 with owner data"},
            {"case": f"other_user_reads_{object_id}", "method": "GET", "path": f"/api/orders/{object_id}", "actor": "different_user", "expected": "403/404 with no data"},
            {"case": f"other_user_updates_{object_id}", "method": "PATCH", "path": f"/api/orders/{object_id}", "actor": "different_user", "expected": "403/404 and unchanged object"},
        ])

    matrix = pd.DataFrame(rows)
    matrix["url"] = base_url.rstrip("/") + matrix["path"]
    return matrix

bola_matrix = build_bola_matrix("https://staging.example.invalid")
bola_matrix

In [ ]:
api_test_plan = {
    "scope": "authorised staging API only",
    "exclude": ["/logout", "/delete", "/admin/production"],
    "assertions": [
        "non-owner receives 403 or 404",
        "response contains no sensitive fields",
        "object state is unchanged",
    ],
    "evidence": ["request", "response status", "actor identity", "object owner"],
}

print(json.dumps(api_test_plan, indent=2))

## 4. SIEM detection and SOAR response

A Wazuh deployment would collect Windows, endpoint, cloud, and application logs. The following rules show the fields that a SOC analyst would want to correlate. The response function is deliberately approval-gated.

In [ ]:
wazuh_rules = [
    {"id": 100001, "level": 10, "description": "Repeated failures followed by a successful logon", "mitre": "T1078"},
    {"id": 100002, "level": 12, "description": "Privileged group membership changed", "mitre": "T1098"},
    {"id": 100003, "level": 12, "description": "Unusual mass file-extension changes", "mitre": "T1486"},
]

pd.DataFrame(wazuh_rules)

In [ ]:
def soar_playbook(event: dict) -> dict:
    required_fields = {"alert_id", "host_id", "severity", "confidence", "analyst_approved"}
    missing_fields = required_fields - event.keys()
    if missing_fields:
        raise ValueError(f"Missing fields: {sorted(missing_fields)}")

    approved = (
        event["severity"] >= 12
        and event["confidence"] >= 0.90
        and event["analyst_approved"]
    )
    return {
        "alert_id": event["alert_id"],
        "decision": "isolate_authorized" if approved else "isolate_pending_approval",
        "actions": ["open incident", "capture triage artifacts", "notify on-call"],
        "rollback": "remove isolation after analyst clearance",
    }

soar_playbook({
    "alert_id": "demo-001",
    "host_id": "LAB-WS-01",
    "severity": 12,
    "confidence": 0.95,
    "analyst_approved": False,
})

## 5. Machine-learning network intrusion baseline

The optional model uses the public KDD Cup 1999 benchmark through scikit-learn. It demonstrates preprocessing and evaluation rather than claiming production accuracy. The download is disabled by default.

In [ ]:
RUN_NIDS_MODEL = False

if RUN_NIDS_MODEL:
    from sklearn.compose import ColumnTransformer
    from sklearn.datasets import fetch_kddcup99
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import classification_report
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

    dataset = fetch_kddcup99(subset="SA", percent10=True, as_frame=True)
    nids_df = dataset.frame.sample(n=min(20000, len(dataset.frame)), random_state=42).copy()
    nids_df.columns = [str(column) for column in nids_df.columns]
    target_column = "target" if "target" in nids_df.columns else nids_df.columns[-1]
    nids_df[target_column] = (
        nids_df[target_column].astype(str).str.strip("b'").str.replace(".", "", regex=False)
    )

    X = nids_df.drop(columns=[target_column])
    y = nids_df[target_column]
    categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_columns = [column for column in X.columns if column not in categorical_columns]

    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_columns),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("one_hot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_columns),
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=120, class_weight="balanced", random_state=42, n_jobs=-1
        )),
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    print(classification_report(y_test, predictions, zero_division=0))
else:
    print("NIDS model is disabled. Set RUN_NIDS_MODEL=True to download and train on the public benchmark.")

## 6. Cloud IAM and DevSecOps controls

These cells document the controls that would be checked with Terraform, Checkov, Trivy, and a dependency scanner. They are templates rather than live cloud changes.

In [ ]:
terraform_public_access_block = r"""resource "aws_s3_bucket_public_access_block" "secure" {
  bucket                  = aws_s3_bucket.data.id
  block_public_acls       = true
  block_public_policy     = true
  ignore_public_acls      = true
  restrict_public_buckets = true
}

# Use workload identity or roles. Do not commit access keys to Terraform files.
"""

checkov_findings = pd.DataFrame([
    ["CKV_AWS_53", "S3 public access block", "high", "Add a public access block"],
    ["CKV_AWS_18", "S3 access logging", "medium", "Enable access logging"],
    ["CKV_SECRET_6", "Hard-coded secret", "critical", "Rotate the secret and use a secret manager"],
], columns=["check_id", "check_name", "severity", "remediation"])

print(terraform_public_access_block)
checkov_findings

In [ ]:
github_actions_security_gate = r"""name: security-gates
on: [push, pull_request]
jobs:
  scan:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: aquasecurity/trivy-action@master
        with:
          scan-type: fs
          severity: CRITICAL,HIGH
          exit-code: '1'
      - run: python -m pip install pip-audit && pip-audit
"""

print(github_actions_security_gate)

## 7. Zero Trust, ransomware risk, and STRIDE

In [ ]:
ztna_controls = pd.DataFrame([
    ["Identity", "MFA with a phishing-resistant factor", "Required"],
    ["Device", "Managed device, EDR, encryption, supported OS", "Required"],
    ["Access", "Per-application policy instead of flat network access", "Required"],
    ["Session", "Short-lived sessions and re-authentication", "Required"],
    ["Visibility", "Tunnel and admin-console events forwarded to the SIEM", "Required"],
    ["Recovery", "Break-glass account stored and tested separately", "Required"],
], columns=["domain", "control", "status"])

risk_register = pd.DataFrame([
    ["R-001", "Identify", "Unmanaged admin accounts", "Privilege abuse expands blast radius", "High"],
    ["R-002", "Protect", "Backups reachable from production", "Backups may be deleted or encrypted", "Critical"],
    ["R-003", "Detect", "No alert on mass file changes", "Containment is delayed", "High"],
    ["R-004", "Respond", "No approved isolation process", "Response actions are inconsistent", "High"],
    ["R-005", "Recover", "Recovery objectives are not tested", "Outage lasts longer than planned", "High"],
], columns=["id", "nist_function", "risk", "impact", "rating"])

stride_threats = pd.DataFrame([
    ["Gateway to Orders API", "Spoofing", "Forged or expired token", "Validate OIDC claims and use short token lifetimes"],
    ["Gateway to Orders API", "Tampering", "Request body changed in transit", "Use TLS and server-side schema validation"],
    ["Orders API to Payment API", "Repudiation", "Actor denies a transaction", "Use correlation IDs and immutable audit logs"],
    ["Orders API to Customer DB", "Information disclosure", "Over-broad query returns PII", "Use least privilege and field filtering"],
    ["Orders API", "Denial of service", "Expensive query flood", "Use rate limits, quotas, and timeouts"],
    ["Orders API", "Elevation of privilege", "Client role claim is trusted", "Enforce authorisation on the server"],
], columns=["flow", "stride_category", "threat", "mitigation"])

ztna_controls, risk_register, stride_threats

## 8. Threat-intelligence aggregation

The functions below normalise indicators and construct provider URLs. API keys are deliberately not stored in the notebook. If this section is extended, use environment variables or a managed secret store and respect provider quotas and terms.

In [ ]:
def normalise_ioc(value: str) -> str:
    value = value.strip().lower()
    if not value:
        raise ValueError("Indicator cannot be empty")
    return value


def indicator_hash(value: str) -> str:
    return hashlib.sha256(normalise_ioc(value).encode()).hexdigest()


def virustotal_search_url(indicator: str) -> str:
    return "https://www.virustotal.com/api/v3/search?query=" + quote(normalise_ioc(indicator), safe="")


def alienvault_ipv4_url(indicator: str) -> str:
    return "https://otx.alienvault.com/api/v1/indicators/IPv4/" + quote(normalise_ioc(indicator), safe="") + "/general"


def merge_iocs(records: list[dict]) -> pd.DataFrame:
    clean_records = []
    for record in records:
        if record.get("indicator") and record.get("source"):
            clean_records.append({
                "indicator": normalise_ioc(record["indicator"]),
                "source": record["source"],
                "confidence": record.get("confidence", "unknown"),
                "first_seen": record.get("first_seen"),
            })
    return pd.DataFrame(clean_records).drop_duplicates(["indicator", "source"])

print(virustotal_search_url("example.com"))
print(alienvault_ipv4_url("203.0.113.10"))

## 9. Export evidence tables

The final cell writes the main evidence tables to the `reports` folder. These files can be attached to a project presentation or used to explain how the notebook produces its outputs.

In [ ]:
exports = {
    "module_scope_matrix.csv": module_scope_matrix,
    "ad_hardening_checklist.csv": ad_hardening_checklist,
    "bola_test_matrix.csv": bola_matrix,
    "wazuh_rules.csv": pd.DataFrame(wazuh_rules),
    "checkov_findings.csv": checkov_findings,
    "ztna_controls.csv": ztna_controls,
    "risk_register.csv": risk_register,
    "stride_threats.csv": stride_threats,
}

for filename, frame in exports.items():
    frame.to_csv(REPORT_DIR / filename, index=False)

print(f"Exported {len(exports)} evidence tables to {REPORT_DIR.resolve()}")

## GitHub deployment

After extraction, copy `index.html` if the GitHub Pages root should open the dashboard. Push the repository and enable Pages from branch `main` and folder `/(root)`. Git and dependency instructions are in `README.md`. Pages serves the website; it does not execute this notebook.


## Conclusion

Cyber Defense Lab pairs a local-first, static dashboard with a defensive security notebook. The notebook covers all 12 dashboard modules and exports eight evidence tables. Live checks remain opt-in and should only be used against authorised targets. GitHub Pages serves the website; run the notebook separately in Jupyter.